# Getting over the dataset

The dataset keeps what is known of every post as JSON files, by platform and profile:
an index, each post, and what `delete` removed.

In [ ]:
import os, tempfile
from pathlib import Path

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault), an inbox and a dataset in a temporary directory.
scratch = Path(tempfile.mkdtemp())
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = str(scratch / "inbox.sqlite3")
os.environ["AMOORION_DATASET"] = str(scratch / "dataset")

In [ ]:
from amoorion import Amoorion

amo = Amoorion().use("dryrun")
first = amo.publish(text="first")[0]["message"]
second = amo.publish(text="second")[0]["message"]
amo.snapshot()

In [ ]:
for path in sorted(amo.dataset.path.rglob("*.json")):
    print(path.relative_to(amo.dataset.path))

`delete` keeps a copy first, and the index remembers:

In [ ]:
amo.delete(id=first["id"])["ok"]
for entry in amo.dataset.index("dryrun", "default").values():
    print(entry.id, entry.state, entry.text)

In [ ]:
amo.dataset.find("dryrun", "default", first["id"]) is not None, sorted(p.name for p in amo.dataset.profile_dir("dryrun", "default").glob("deleted/*"))

It is plain files, so anything reads it, a search index of your own included:

In [ ]:
import json

index = json.loads((amo.dataset.profile_dir("dryrun", "default") / "index.json").read_text())
[(post["id"], post["state"]) for post in index["posts"]]